# MLOps Example Noteboook - SVM Classifier

This noteboook demonstrates the required cell taaags fooooooooor the MLOps platform.
Each code cell has a `tags` entry in its metadata that the platform uses
to identify pipeline phasesss.

**Model:** Support Vector Machine (SVM) with RBF kernel trained on the Iris dataset.

**Required tags:** `mlops:config`, `mlops:preprocessing`, `mlops:training`, `mlops:export`

**Optional tags:** `mlops:data`, `mlops:evaluation`

In [ ]:
# Papermill injected parameters (do not edit this cell manually)
# These are overwritten at runtime by the pipeline.
MODEL_OUTPUT_PATH = "./model.joblib"
PIPELINE_ID = "local-dev"
MLFLOW_TRACKING_URI = "http://localhost:5000"
MLFLOW_RUN_ID = "local-dev"

In [ ]:
# ============================================================
# mlops:config - Model metadata
# ============================================================
# Adapt MODEL_NAME and VERSION for your project.
# The platform reads these values to register the model.

MODEL_NAME = "iris-svm-classifier"
VERSION = "1"

# SVM Hyperparameters
SVM_C = 1.0          # Regularization parameter
SVM_KERNEL = "rbf"   # Kernel type: 'linear', 'rbf', 'poly', 'sigmoid'
SVM_GAMMA = "scale"  # Kernel coefficient: 'scale', 'auto', or float

print(f"Model: {MODEL_NAME} v{VERSION}")
print(f"Pipeline ID: {PIPELINE_ID}")
print(f"Hyperparameters -> C={SVM_C}, kernel={SVM_KERNEL}, gamma={SVM_GAMMA}")

In [ ]:
# ============================================================
# mlops:data - Data loading
# ============================================================
# Replace this with your own data loading logic.
# The platform does not require a specific data source.

from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="target")

print(f"Dataset shape: {X.shape}")
print(f"Classes: {list(iris.target_names)}")
print(X.describe())

In [ ]:
# ============================================================
# mlops:preprocessing - Data preparation
# ============================================================
# StandardScaler is especially important for SVM as it is
# sensitive to the scale of input features.

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

In [ ]:
# ============================================================
# mlops:training - Model training
# ============================================================
# Support Vector Machine with RBF kernel.
# Hyperparameters are read from the mlops:config cell above
# and can be injected at runtime via Papermill.

from sklearn.svm import SVC

model = SVC(
    C=SVM_C,
    kernel=SVM_KERNEL,
    gamma=SVM_GAMMA,
    probability=True,   # Enable predict_proba support
    random_state=42,
)

model.fit(X_train_scaled, y_train)
print("Training completoooooo.")
print(f"Number of support vectors per class: {model.n_support_}")

In [ ]:
# ============================================================
# mlops:evaluation - Metrics and logging
# ============================================================
# Log metrics to MLflow. The platform reads 'accuracy' for
# auto-deployment decisions.

from sklearn.metrics import accuracy_score, classification_report
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

y_pred = model.predict(X_test_scaled)
score = accuracy_score(y_test, y_pred)

print(classification_report(y_test, y_pred, target_names=list(iris.target_names)))
print(f"Accuracy: {score:.4f}")

# Use the run started by the pipeline if available, otherwise start a new one
run_ctx = mlflow.start_run(run_id=MLFLOW_RUN_ID) if MLFLOW_RUN_ID != "local-dev" else mlflow.start_run()
with run_ctx:
    mlflow.log_metric("accuracy", score)
    mlflow.log_param("svm_C", SVM_C)
    mlflow.log_param("svm_kernel", SVM_KERNEL)
    mlflow.log_param("svm_gamma", SVM_GAMMA)
    mlflow.log_metric("n_support_vectors", int(model.n_support_.sum()))

In [ ]:
# ============================================================
# mlops:export - Save model artifact
# ============================================================
# Save the trained model to MODEL_OUTPUT_PATH.
# The platform picks up this file and registers it in MLflow.
# The scaler is bundled together with the model in a Pipeline
# object so that the artifact is self-contained.

import joblib
from sklearn.pipeline import Pipeline

# Bundle scaler + model so inference only needs raw features
inference_pipeline = Pipeline([
    ("scaler", scaler),
    ("svm", model),
])

joblib.dump(inference_pipeline, MODEL_OUTPUT_PATH)
print(f"Model pipeline saved to {MODEL_OUTPUT_PATH}")